In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.functions import left

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

flag_run_task = dbutils.jobs.taskValues.get(taskKey = "00_FILE_INGEST", key = "run_tsk")

if flag_run_task:
    dt_ref_silver = spark.table("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]

    df_silver_bndes = spark.table("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto").filter(col("data_ref").isin(dt_ref_silver))

    df_gold = df_silver_bndes

    df_gold_bndes_write = df_gold.groupBy(
        df_gold.data_ref,
        substring(df_gold.dt_contrato,1,7).alias("anomes_contrato"),
        df_gold.sigla_uf,
        df_gold.desc_municipio,
        df_gold.desc_orig_recurso,
        df_gold.situ_contrato,
        df_gold.desc_garantia,
        df_gold.if_nome,
        df_gold.porte,
        df_gold.desc_setor_bndes,
        df_gold.desc_cnae,
        df_gold.desc_area_op,
        df_gold.desc_inova,
        df_gold.desc_produto,
        df_gold.desc_forma_apoio,
        df_gold.desc_modalidade_apoio
    ).agg(
        count(df_gold.cod_contrato_seq).alias("qt_contrato_seq"),
        avg(regexp_replace(col("pct_juros"), "%","").cast("float")).alias("avg_pct_juros"),
        sum(df_gold.vlr_contrato).alias("sum_vlr_contrato")
    )

    try:
        dt_ref_gold = spark.table("data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]
        dt_ref_update = df_gold_bndes_write.agg(max(col("data_ref"))).first()[0]

        if dt_ref_update > dt_ref_gold:
            try:
                print("appendando base atualizada")
                df_gold_bndes_write.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto")
            except:
                print("erro ao appendar")
        elif dt_ref_update == dt_ref_gold:
            print("erro: base com referencia igual a ultima atualização")
        else:
            print("erro: base com referencia anterior a ultima atualização")
    except:
        print("criando base")
        df_gold_bndes_write.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto")
else: print(flag_run_task)